In [1]:
import glob, zarr, os, napari
import dask.array as da
from tqdm.auto import tqdm
from natsort import natsorted
import napari
import pandas as pd
SCALE_TUPLE = (0.165, 0.165)
SPLIT_IMAGE_KWS =  ['top', 'bot', 'left', 'right']

# Generating full image FOVs

In [2]:
zarr_fns = glob.glob('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep*/mouse_*/zarr/*max_proj.zarr')
# Filter and sort
zarr_fns = natsorted([
                        fn for fn in zarr_fns
                        if not any(x in fn for x in SPLIT_IMAGE_KWS)])
zarr_fns

['/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_1/zarr/rep_1_mouse_1_max_proj.zarr',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_2/zarr/rep_1_mouse_2_max_proj.zarr',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_3/zarr/rep_1_mouse_3_max_proj.zarr',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_4/zarr/rep_1_mouse_4_max_proj.zarr',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_5/zarr/rep_1_mouse_5_max_proj.zarr',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_1/zarr/rep_2_mouse_1_max_proj.zarr',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_2/zarr/rep_2_mouse_2_max_proj.zarr',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_7/zarr/rep_2_mouse_7_max_proj.zarr',
 '/mnt/O

In [3]:
df_metadata = pd.read_pickle('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/results/arx/metadata.pkl')

In [4]:
df_metadata

,mouse_num,replicate,condition,basename_stem
0,1,1,H2O,20250815_40X_TimerMtb_BP_mice1_DAPI_TimerG_Tim...
1,2,1,DMSO,20250901_40X_TimerMtb_BP_mice2_mice3_mice4_DAP...
2,3,1,RIF,20250901_40X_TimerMtb_BP_mice2_mice3_mice4_DAP...
3,4,1,RIF,20250901_40X_TimerMtb_BP_mice2_mice3_mice4_DAP...
4,5,1,RIF,20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAP...
5,6,1,RIF,20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAP...
6,7,1,PZA,20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAP...
7,8,1,PZA,20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_Time...
8,9,1,PZA,20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_Time...
9,10,1,PZA,20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_Ti...


In [5]:
zarr_fn = zarr_fns[5]
print(zarr_fn)

/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_1/zarr/rep_2_mouse_1_max_proj.zarr


In [6]:
import napari
import glob
import os
import json
import re
import dask.array as da
import pandas as pd
from tqdm.auto import tqdm
from pathlib import Path

# Setup QC directory
qc_dir = Path("/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/results/tissue_figures")
qc_dir.mkdir(exist_ok=True)

mtb_colormap = {1: '#FFFF00'} # yellow: #FFFF00

# Regex to parse info from filename
rep_pattern = re.compile(r"rep_(\d+)")
mouse_pattern = re.compile(r"mouse_(\d+)")
    


# --- Parse Identity ---
basename = os.path.basename(os.path.normpath(zarr_fn))

rep_match = rep_pattern.search(basename)
rep_id = int(rep_match.group(1)) if rep_match else None

mouse_match = mouse_pattern.search(basename)
mouse_id = int(mouse_match.group(1)) if mouse_match else None
# --- Lookup Condition ---
condition = "Unknown"
if rep_id is not None and mouse_id is not None:
    # Query the metadata df for the matching row
    # keys: mouse_num, replicate, condition
    match = df_metadata[
        (df_metadata['mouse_num'] == mouse_id) & 
        (df_metadata['replicate'] == rep_id)
    ]
    if not match.empty:
        condition = match.iloc[0]['condition']

# --- Load Data ---
pyramid_image_stack = [da.from_zarr(fn) for fn in glob.glob(f"{zarr_fn}/s*")]
cyto_masks = da.from_zarr(f'{zarr_fn}/labels/cyto_seg/0')
mtb_masks = da.from_zarr(f'{zarr_fn}/labels/ground_truth_mtb/0')

# --- Viewer Setup ---
viewer = napari.Viewer(title=basename)

viewer.add_image(pyramid_image_stack, channel_axis=0, 
                 colormap=['blue', 'green', 'magenta'], scale=SCALE_TUPLE)
viewer.add_labels(cyto_masks, name='cytoplasm', scale=SCALE_TUPLE,)# contour=4)
viewer.add_labels(mtb_masks, name='mtb', scale=SCALE_TUPLE, 
                  colormap=mtb_colormap,) #contour=4)

viewer.scale_bar.visible = True
viewer.scale_bar.unit = "um"

# --- Screenshot Keybinding ---
@viewer.bind_key('s')
def save_qc_screenshot(viewer):
    # Define base pattern
    base_name_str = f"{basename}_{condition}_QC"
    
    # Check for existence and increment counter if needed
    counter = 0
    candidate_name = base_name_str
    while (qc_dir / f"{candidate_name}.png").exists():
        counter += 1
        candidate_name = f"{base_name_str}_{counter}"
    
    # 1. Save Image
    image_path = qc_dir / f"{candidate_name}.png"
    viewer.screenshot(path=str(image_path))
    
    # 2. Save Metadata
    meta = {
        "file": basename,
        "condition": condition,
        "mouse": mouse_id,
        "rep": rep_id,
        "zoom": viewer.camera.zoom,
        "center": list(viewer.camera.center),
        "layers": {l.name: l.visible for l in viewer.layers},
        "save_iteration": counter
    }
    
    json_path = qc_dir / f"{candidate_name}.json"
    with open(json_path, 'w') as f:
        json.dump(meta, f, indent=4)
        
    viewer.status = f"Saved: {candidate_name}"
    print(f"Captured: {candidate_name}")


/home/dayn/miniconda3/envs/macrohet_fixed/lib/python3.10/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (43776, 35482) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(
/home/dayn/miniconda3/envs/macrohet_fixed/lib/python3.10/site-packages/napari/utils/colormaps/colormap.py:455: UserWarning: color_dict did not provide a default color. Missing keys will be transparent. To provide a default color, use the key `None`, or provide a defaultdict instance.
  warn(
/home/dayn/miniconda3/envs/macrohet_fixed/lib/python3.10/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (45850, 52070) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(
Traceback (most recent call last):
  File "/home/dayn/miniconda3/envs/macrohet_fixed/lib/python3.10/site-packages/napari/_qt/widgets/qt_vie

In [7]:
# --- Lookup Condition ---
condition = "Unknown"
if rep_id is not None and mouse_id is not None:
    # Query the metadata df for the matching row
    # keys: mouse_num, replicate, condition
    match = df_metadata[
        (df_metadata['mouse_num'] == mouse_id) & 
        (df_metadata['replicate'] == rep_id)
    ]
    if not match.empty:
        condition = match.iloc[0]['condition']


In [ ]:
viewer.scale_bar.

In [17]:
viewer.scale_bar.font_size = 0  # default is 10


Captured: rep_2_mouse_1_max_proj.zarr_H2O_QC_20


In [ ]:
 "zoom": 6.360214222434655,
    "center": [
        0.0,
        3080.904339020486,
        2191.582809159516
    ],

In [18]:
viewer.camera.center = (0, 3080.904339020486,
        2191.582809159516)

In [19]:
viewer.camera.zoom = 6.360214222434655

Captured: rep_2_mouse_1_max_proj.zarr_H2O_QC_21


### started at 1705

In [19]:
from datetime import datetime

# Get and print current time
print(datetime.now())

# Optional: Print just the time (HH:MM:SS)
print(datetime.now().strftime("%H:%M:%S"))

2025-12-22 11:22:02.779232
11:22:02
